# scikit-learn — the parts you'll actually use

> Pipelines, cross-validation, and the five models that solve 80 % of
> tabular problems.

scikit-learn is enormous. Working ML code uses roughly 10 % of it. This is
that 10 %, demonstrated on California housing — a real, messy, mid-sized
dataset where every preprocessing decision visibly moves the score.

## 1. The dataset

8 numeric features (median income, house age, average rooms, population,
latitude, longitude…), one continuous target (median house value), 20 640
rows. Standard sklearn import, no extra download.

In [1]:
import numpy as np, pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

data = fetch_california_housing(as_frame=True)
X, y = data.data, data.target
print(X.shape, '→', y.shape)
X.head()

(20640, 8) → (20640,)


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


In [2]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=0)
print(f'train: {len(Xtr):>5}   test: {len(Xte):>5}')

train: 16512   test:  4128


## 2. Pipeline — the single object that holds preprocessing + model

The Pipeline is the single most underused class in sklearn. It bundles
preprocessing and the model into one estimator, which means:

- one `.fit()` call, one `.predict()` call
- cross-validation sees the preprocessing too (no leakage)
- model persistence saves the full pipeline (no "I forgot to scale" bugs)
- swapping models is one line

In [3]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

pipe = Pipeline([
    ('scale', StandardScaler()),
    ('model', LinearRegression()),
])
pipe.fit(Xtr, ytr)
print(f'train R²  {pipe.score(Xtr, ytr):.3f}')
print(f'test R²   {pipe.score(Xte, yte):.3f}')

train R²  0.609
test R²   0.594


/Users/allamaprabhuani/miniconda3/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/allamaprabhuani/miniconda3/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/allamaprabhuani/miniconda3/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
/Users/allamaprabhuani/miniconda3/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/allamaprabhuani/miniconda3/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/allamaprabhuani/miniconda3/lib/python3.10/site-packages/sklearn/li

## 3. Five models that solve most tabular problems

You will reach for one of these in 90 % of tabular ML work. Keep them in
your head; reach for fancier models only when these plateau.

In [4]:
from sklearn.linear_model import Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

models = {
    'linear': Pipeline([('sc', StandardScaler()), ('m', LinearRegression())]),
    'ridge':  Pipeline([('sc', StandardScaler()), ('m', Ridge(alpha=1.0))]),
    'lasso':  Pipeline([('sc', StandardScaler()), ('m', Lasso(alpha=0.01, max_iter=10000))]),
    'tree':   DecisionTreeRegressor(max_depth=8, random_state=0),
    'rf':     RandomForestRegressor(n_estimators=120, max_depth=14, n_jobs=-1, random_state=0),
    'gb':     GradientBoostingRegressor(n_estimators=200, max_depth=4, random_state=0),
}

## 4. `cross_val_score` — why a single split lies

A single train/test split gives you one number. That number could be lucky
or unlucky. **k-fold cross-validation** rotates the test fold across the
data and gives you a *distribution* of scores — that's the honest signal.

In [5]:
from sklearn.model_selection import cross_val_score

scores = {name: cross_val_score(m, Xtr, ytr, cv=5, scoring='r2', n_jobs=-1)
          for name, m in models.items()}
for name, s in scores.items():
    print(f'{name:>7s}  R² = {s.mean():.3f}  ± {s.std():.3f}')

/Users/allamaprabhuani/miniconda3/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/allamaprabhuani/miniconda3/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/allamaprabhuani/miniconda3/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/allamaprabhuani/miniconda3/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/allamaprabhuani/miniconda3/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
/Users/allamaprabhuani/miniconda3/lib/python3.10/site-packages/sklearn/li

/Users/allamaprabhuani/miniconda3/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/allamaprabhuani/miniconda3/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/allamaprabhuani/miniconda3/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/allamaprabhuani/miniconda3/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/allamaprabhuani/miniconda3/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/allamaprabhuani/miniconda3/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/allamaprabhuani/miniconda3/lib/python3.10/site

/Users/allamaprabhuani/miniconda3/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/allamaprabhuani/miniconda3/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/allamaprabhuani/miniconda3/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
/Users/allamaprabhuani/miniconda3/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/allamaprabhuani/miniconda3/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/allamaprabhuani/miniconda3/lib/python3.10/site-packages/sklearn/li

 linear  R² = 0.605  ± 0.006
  ridge  R² = 0.605  ± 0.006
  lasso  R² = 0.603  ± 0.006
   tree  R² = 0.678  ± 0.007
     rf  R² = 0.797  ± 0.005
     gb  R² = 0.821  ± 0.005


## 5. `GridSearchCV` vs `RandomizedSearchCV`

`GridSearchCV` tries every combination — exhaustive, expensive.
`RandomizedSearchCV` samples random combinations — much faster, almost as
good for >3 hyperparameters. **Default to Randomized.**

In [6]:
from sklearn.model_selection import GridSearchCV

grid = {
    'm__alpha': [0.01, 0.1, 1.0, 10.0, 100.0],
}
gs = GridSearchCV(
    Pipeline([('sc', StandardScaler()), ('m', Ridge())]),
    param_grid=grid, cv=5, scoring='r2', n_jobs=-1
)
gs.fit(Xtr, ytr)
print(f'best alpha: {gs.best_params_["m__alpha"]}')
print(f'best CV R²: {gs.best_score_:.3f}')
print(f'test R²:    {gs.score(Xte, yte):.3f}')

/Users/allamaprabhuani/miniconda3/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/allamaprabhuani/miniconda3/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/allamaprabhuani/miniconda3/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/allamaprabhuani/miniconda3/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/allamaprabhuani/miniconda3/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/allamaprabhuani/miniconda3/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: invalid value encountered in matmul
  retur

best alpha: 10.0
best CV R²: 0.605
test R²:    0.594


/Users/allamaprabhuani/miniconda3/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/allamaprabhuani/miniconda3/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/allamaprabhuani/miniconda3/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/allamaprabhuani/miniconda3/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/allamaprabhuani/miniconda3/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/allamaprabhuani/miniconda3/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: invalid value encountered in matmul
  retur

## 6. Saving and loading without breaking versioning

`joblib.dump()` serialises the whole pipeline. Always pin your
scikit-learn version — pickle files break across major versions.

In [7]:
import joblib, sklearn
gb = models['gb'].fit(Xtr, ytr)
joblib.dump({'pipe': gb, 'sklearn_version': sklearn.__version__}, '/tmp/gb_model.joblib')

loaded = joblib.load('/tmp/gb_model.joblib')
print(f'saved with sklearn {loaded["sklearn_version"]}')
print(f'loaded test R²: {loaded["pipe"].score(Xte, yte):.3f}')

saved with sklearn 1.7.2
loaded test R²: 0.826


## What you've built

- A Pipeline that holds preprocessing + model as one object.
- A 5-model bake-off with honest cross-validated scores.
- A `GridSearchCV` over a real hyperparameter.
- A persisted model that survives Python sessions.

**Next:** [`03 — Introduction to Neural Networks`](/tutorials/03-neural-networks-intro/)
covers what's different when the model is a network and the framework is
PyTorch.

For the rest of the series, see [tutorials](/tutorials/).